# 🔴 Solution: Closest Pair of Points

**Algorithm:** Divide and conquer — O(N log N)

**Reduction:** Sort by x, recurse for δ, then check a strip of width 2δ around the midline sorted by y — at most 7 neighbors per point suffice (geometric packing argument).

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# algorithm: divide-and-conquer O(N log N)

import numpy as np

def closest_pair(points):
    pts = np.array(points, dtype=float)

    def _dist(a, b):
        return np.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

    def _brute(p, idx):
        best=float('inf'); bi=bj=-1
        for ii in range(len(p)):
            for jj in range(ii+1, len(p)):
                d=_dist(p[ii], p[jj])
                if d < best: best=d; bi,bj=idx[ii],idx[jj]
        return best, bi, bj

    def _rec(p, idx):
        n=len(p)
        if n<=3: return _brute(p, idx)
        mid=n//2; mx=p[mid,0]
        dl,il,jl=_rec(p[:mid], idx[:mid])
        dr,ir,jr=_rec(p[mid:], idx[mid:])
        if dl<=dr: delta,bi,bj=dl,il,jl
        else:      delta,bi,bj=dr,ir,jr
        mask=np.abs(p[:,0]-mx)<delta
        sp=p[mask]; si=idx[mask]; so=np.argsort(sp[:,1])
        sp=sp[so]; si=si[so]
        for k in range(len(sp)):
            for l in range(k+1, min(k+8, len(sp))):
                if sp[l,1]-sp[k,1]>=delta: break
                d=_dist(sp[k],sp[l])
                if d<delta: delta=d; bi,bj=si[k],si[l]
        return delta, bi, bj

    order=np.argsort(pts[:,0])
    d,i,j=_rec(pts[order], order)
    if i>j: i,j=j,i
    return float(d), int(i), int(j)

In [ ]:
# 🔍 Verify solution
# Simple: distance 5 (3-4-5 triangle)
pts = np.array([[0.,0.],[3.,4.],[1.,0.]])
dist, i, j = closest_pair(pts)
print(f"distance={dist:.4f}, i={i}, j={j}")  # expect 1.0, indices 0 and 2

# Grid
pts2 = np.array([[0.,0.],[1.,0.],[0.,1.],[1.,1.]])
d,i,j = closest_pair(pts2)
print(f"grid: dist={d:.4f}, ({i},{j})")       # expect 1.0

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: two points ─────────────────────────────────────────────────────
pts = np.array([[0.,0.],[3.,4.]])
d,i,j = closest_pair(pts)
assert abs(d-5.0)<1e-9 and i==0 and j==1, f"dist={d}, ({i},{j})"
print("Test 1 passed: two points")

# ── Test 2: three collinear ────────────────────────────────────────────────
pts2 = np.array([[0.,0.],[1.,0.],[10.,0.]])
d2,i2,j2 = closest_pair(pts2)
assert abs(d2-1.0)<1e-9 and set([i2,j2])=={0,1}, f"dist={d2}, ({i2},{j2})"
print("Test 2 passed: three collinear")

# ── Test 3: brute-force verification ──────────────────────────────────────
rng=np.random.default_rng(42); pts3=rng.uniform(-10,10,(60,2))
best=float('inf'); bi=bj=-1
for ii in range(len(pts3)):
    for jj in range(ii+1,len(pts3)):
        dd=np.linalg.norm(pts3[ii]-pts3[jj])
        if dd<best: best=dd; bi,bj=ii,jj
d3,i3,j3=closest_pair(pts3)
assert abs(d3-best)<1e-9, f"dist mismatch: {d3} vs {best}"
assert set([i3,j3])==set([bi,bj]), f"index mismatch"
print("Test 3 passed: matches brute-force on 60 points")

# ── Test 4: duplicates → distance=0 ──────────────────────────────────────
pts4=np.array([[1.,2.],[5.,3.],[1.,2.],[9.,8.]])
d4,i4,j4=closest_pair(pts4)
assert abs(d4-0.)<1e-12 and i4<j4, f"duplicate: dist={d4}, ({i4},{j4})"
print("Test 4 passed: duplicate points → distance=0")

# ── Test 5: large N=10000 ─────────────────────────────────────────────────
rng2=np.random.default_rng(99); pts5=rng2.uniform(-1000,1000,(10000,2))
t0=time.time(); d5,i5,j5=closest_pair(pts5); elapsed=time.time()-t0
assert d5>0 and 0<=i5<j5<10000
assert elapsed<5.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: N=10000 ({elapsed:.3f}s)")

print("\nAll tests passed!")